# 01_prepare_dataset

Este notebook prepara el dataset por capas Medallion.

Flujo aplicado:

1. Definir ruta local de dataset.
2. Descargar/subir manualmente los ficheros raw a esa ruta.
3. Preparar contexto (`main` -> rama `transform/v[version]`).
4. Ejecutar Bronze.
5. Ejecutar Silver (`captia_points.parquet`) y validar contrato.
6. Ejecutar Gold (`train.parquet` y `test.parquet`) y validar.
7. Finalizar (merge + tag de versión) solo si hubo commits de capa.

In [ ]:
import os
import getpass
import lakefs_sdk
import sys
from mlops.utils import LakeFSManager
from pathlib import Path
from tempfile import gettempdir

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = next(
    (candidate for candidate in _candidates if (candidate / "src").exists()),
    Path.cwd(),
)
SRC_PATH = Path(os.getenv("PYTHONPATH"))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

lakefs = LakeFSManager()

## 1. Ruta de datos de entrada

Descarga o coloca manualmente los ficheros raw del dataset en `RAW_DATA_DIR` antes de ejecutar la preparación.

In [ ]:
URL = # url del dataset
RAW_DATA_DIR = Path(
    os.getenv("RAW_DATA_DIR", str(PROJECT_ROOT / "data"))
)
TMP_DIR = Path(os.getenv("TMP_DIR", gettempdir()))
TMP_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = os.getenv("DATASET_NAME") or None
CASE_ID = os.getenv("CASE_ID") or None
CREATED_BY = os.getenv("CREATED_BY") or getpass.getuser()
if not CREATED_BY:
    raise ValueError("No se pudo resolver CREATED_BY")

print("URL: ", URL)
print("RAW_DATA_DIR:", RAW_DATA_DIR.resolve())
print("TMP_DIR:", TMP_DIR.resolve())
print("DATASET_NAME:", DATASET_NAME)
print("CASE_ID:", CASE_ID)
print("CREATED_BY:", CREATED_BY)

In [ ]:
import zipfile
import os

ZIP = TMP_DIR / "dataset.zip"

!wget $URL -O $ZIP

# Unzip the file
with zipfile.ZipFile(ZIP, 'r') as zip_ref:
    zip_ref.extractall(RAW_DATA_DIR)

os.remove(ZIP)

# List extracted files to verify
print(os.listdir(RAW_DATA_DIR))

## 2. Bronze

Esta sección debe limitarse a versionar los ficheros raw originales, sin transformar su contenido.

In [ ]:
branch = lakefs.new_transform_branch(dataset=DATASET_NAME, case_id=CASE_ID)
bronze_commit = lakefs.commit_bronze(
    os.listdir(RAW_DATA_DIR),
    dataset=DATASET_NAME,
    case_id=CASE_ID,
    branch=branch
)
print("bronze_commit: ", bronze_commit)

## 3. Silver

Esta sección debe construir el dataframe tratado/limpio en notebook y generar `captia_points.parquet` cumpliendo `captia_schema.json`.

In [ ]:
# Aquí se realiza el tratamiento de los datos, limpieza y se crea un único dataframe limpio

In [ ]:
prepared_df = # dataframe limpio

silver_commit, silver_quality = lakefs.commit_silver(
    prepared_df,
    dataset=DATASET_NAME,
    case_id=CASE_ID,
    branch=branch
)
print("silver_commit: ", silver_commit)
print("silver_rows: ", silver_quality.get("rows"))

## 4. Gold

Esta sección debe crear `train.parquet` y `test.parquet`, junto con informes/metadatos, validando que los datos necesarios están presentes.

In [ ]:
# split train y test

gold_commit, gold_quality = lakefs.commit_gold(
    train_df, test_df
    dataset=DATASET_NAME,
    case_id=CASE_ID,
    branch=branch
)
print("gold_commit: ", gold_commit)
print("gold_quality: ", gold_quality.get("rows"))

## 5. Finalización

In [ ]:
result = pipeline.finalize_preparation(
    context,
    commits={
        "bronze": bronze_commit,
        "silver": silver_commit,
        "gold": gold_commit,
    },
    bronze_change_report=bronze_change_report,
    silver_change_report=silver_change_report,
    gold_change_report=gold_change_report,
    silver_quality=silver_quality,
    gold_quality=gold_quality,
    split_details=split_details,
)

print("status:", result["status"])
print("version:", result.get("version"))
print("branch:", result.get("branch"))
print("commits:", result.get("commits"))

In [ ]:
change_report = result.get("change_report", {})
for layer in ("bronze", "silver", "gold"):
    layer_report = change_report.get(layer, {})
    print(layer, "has_changes:", layer_report.get("has_changes"))

print("training dataset:", result.get("dataset"))
print("target:", result.get("training_input", {}).get("target"))